## Evaluate Ukrainian OVDP

**TODO**:
- SIM method
- branch logic

### Bond: YTM Method

In [ ]:
from datetime import datetime
import requests
import pandas as pd
import QuantLib as ql

def price_from_yield(calc_date: ql.Date, target_yield: float, bond_data: dict):
    ql.Settings.instance().evaluationDate = calc_date
     
    # ======================= КУПОННІ ДАТИ =======================
    def coupon_dates_from_maturity(bond_data: dict) -> ql.Date:
        """maturity, maturity-182, maturity-2*182, ... (повертає ВОЗР. список)."""
        maturity = bond_data['maturity_date']
        until_date = bond_data['issue_date']
        d = maturity
        desc = [d]
        for _ in range(120):  # safety
            d = d - bond_data['payment_frequency']
            desc.append(d)
            if d <= until_date:
                break
        asc = sorted(set(desc))
        return asc[0]

    # розрахувати зміщену дату effectiveDate для вирівнювання 1-го купону
    start_date = coupon_dates_from_maturity(bond_data)    
    
    # Правильні конвенції для українського суверенного боргу
    # day_count = ql.ActualActual(ql.ActualActual.Bond) # day_count = ql.ActualActual(ql.ActualActual.ISMA) # day_count = ql.Actual364()
    day_count = ql.ActualActual(ql.ActualActual.ISMA)
    calendar = ql.NullCalendar()
    business_convention = ql.Unadjusted

    # Точний розклад купонів
    schedule = ql.MakeSchedule(
        # effectiveDate=bond_data['issue_date'],
        effectiveDate=start_date,
        terminationDate=bond_data['maturity_date'],
        tenor=ql.Period(bond_data['payment_frequency'], ql.Days),
        calendar=calendar,
        convention=business_convention,
        terminalDateConvention=business_convention,
        rule=ql.DateGeneration.Backward, # Дуже важливо для ОВДП Мінфіну
        endOfMonth=False
    )

    # Ініціалізація грошових потоків та базового класу Bond # bond = ql.Bond(settlement_days, calendar, issue_date, fixed_rate_leg)
    fixed_rate_leg = ql.FixedRateLeg(
        schedule,
        day_count,
        [bond_data['face_value']],
        [bond_data['coupon_rate']]
    )

    settlement_days = 0
    bond = ql.FixedRateBond(
        settlement_days,
        bond_data['face_value'],
        schedule,
        [bond_data['coupon_rate']],
        day_count,
        business_convention
    )

    # Свторення явної кривої дисконтування та прив'язка bond до PricingEngine
    def get_dfs(calc_date, fixed_rate_leg, target_yield):
        dates = []
        dates.append(calc_date)
        discount_factors = []
        discount_factors.append(1.0)
        future = [d.date() for d in fixed_rate_leg if d.hasOccurred() == False]
        for d in future:
            days = d - calc_date
            t = max(0.0, days / 365.0)
            discount_factor = 1 / ((1.0 + target_yield) ** t)
            dates.append(d)
            discount_factors.append(discount_factor)
        return dates, discount_factors

    dates, discount_factors = get_dfs(calc_date, fixed_rate_leg, target_yield)
    discount_curve = ql.DiscountCurve(dates, discount_factors, day_count, calendar)
    term_structure_handle = ql.YieldTermStructureHandle(discount_curve)
    bond_engine = ql.DiscountingBondEngine(term_structure_handle)
    bond.setPricingEngine(bond_engine)

    # 6. Розрахунок та виведення результатів
    settlement_date = bond.settlementDate().ISO()
    dirty_drice = round(bond.dirtyPrice() / 100 * bond_data['face_value'], 2)
    clean_price = round(bond.cleanPrice() / 100 * bond_data['face_value'], 2)
    accrued_amount = round(bond.accruedAmount() / 100 * bond_data['face_value'], 2)    

    # print(f"Дата розрахунку: {bond.settlementDate().ISO()}; Дата випуску: {bond_data['issue_date'].ISO()}; Теоретична дата початку: {bond_data['start_date'].ISO()}")
    print(f"Дата розрахунку: {bond.settlementDate().ISO()}; isin: {bond_data['isin']}; Dirty: {bond.dirtyPrice() / 100 * bond_data['face_value']:.2f} UAH; НКД: {bond.accruedAmount() / 100 * bond_data['face_value']:.2f} | Clean: {bond.cleanPrice() / 100 * bond_data['face_value']:.2f}")
    # print()
    # print("Графік купонів:")
    # for cf in fixed_rate_leg:
    #     # if cf.hasOccurred() == False:
    #         coupon = ql.as_coupon(cf)
    #         if coupon:
    #             leg_item = ql.as_coupon(cf)
    #             start_date = leg_item.accrualStartDate()
    #             end_date = leg_item.accrualEndDate()
    #             days = end_date - start_date
    #             amount = leg_item.amount()
    #             print(f"{start_date.ISO()} | {end_date.ISO()} | {days} | {amount:.2f}")

    # print()
    # print("Дисконт-фактори:")
    # for node in discount_curve.nodes():
    #     print(f"{node[0].ISO()} | {node[1]}")
    
    return {
        'start_date':start_date.ISO(),
        'settlement_date': settlement_date,
        'dirty_drice': dirty_drice,
        'clean_price': clean_price,
        'accrued_amount': accrued_amount
    }
    # End function

## Start
url = f"https://bank.gov.ua/depo_securities?json"
response = requests.get(url=url)
data = response.json()
df = pd.DataFrame(data)
df.to_excel("data/ua-bonds.xlsx", index=False, sheet_name="ua-bonds", engine='openpyxl')

# TODO
# Параметри розрахунків
calc_date = ql.Date(6, 10, 2026)
target_yield = 0.16
# bond_data = {
#     'isin': 'UA4000239131',
#     'face_value': 1000.0,
#     'currency': 'UAH',
#     'coupon_rate': 0.1649,
#     'issue_date': ql.Date('06.10.2026', "%d.%m.%Y"),
#     'maturity_date': ql.Date('21.03.2029', "%d.%m.%Y"),
#     'payment_frequency': 182,
#     'coupon_per_year': 2
# }

# price_from_yield(calc_date, target_yield, bond_data)

print("Обробку розпочато.")
dataframes = []
dataset = pd.read_excel(f"data/ua-bonds.xlsx", sheet_name="ua-bonds", engine='openpyxl')
for row in dataset.itertuples():
    bond_data = {
        'isin': row.cpcode,
        'face_value': float(row.nominal),
        'currency': row.val_code,
        'coupon_rate': round(float(row.auk_proc) / 100, 4),
        'issue_date': ql.Date(row.razm_date, "%Y-%m-%d"),
        'maturity_date': ql.Date(row.pgs_date, "%Y-%m-%d"),
        'payment_frequency': int(row.pay_period),
        'coupon_per_year': 365 // int(row.pay_period),
        'bondtype': row.cptype,
        'bondcpdescr': row.cpdescr,
        'cptype_nkcpfr': row.cptype_nkcpfr,
        'cpcode_cfi': row.cpcode_cfi,
    }
    if all([bond_data['maturity_date'] >= calc_date, bond_data['bondtype'] == 'DCP', bond_data['cptype_nkcpfr'] in [1120240, 1120244]]):
        # print(f"   isin: {bond_data['isin']}")
        pfy_data = price_from_yield(calc_date, target_yield, bond_data)
        bond_data.update(pfy_data)
        bond_data['issue_date'] = bond_data['issue_date'].ISO()
        bond_data['maturity_date'] = bond_data['maturity_date'].ISO()
        dataframes.append(bond_data)

result = pd.DataFrame(dataframes)
with pd.ExcelWriter(f"data/ua-bonds.xlsx", mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
    result.to_excel(writer, sheet_name="ua-bonds-pfy", index=False)
print("Обробку завершено.")
# TODO

### Bond: YTM & SIM Method DEV

In [ ]:
from datetime import datetime
import requests
import pandas as pd
import QuantLib as ql

def price_from_yield(calc_date: ql.Date, target_yield: float, bond_data: dict):
    ql.Settings.instance().evaluationDate = calc_date   

    # ======================= КУПОННІ ДАТИ =======================
    def coupon_dates_from_maturity(bond_data: dict) -> ql.Date:
        """maturity, maturity-182, maturity-2*182, ... (повертає ВОЗР. список)."""
        maturity = bond_data['maturity_date']
        until_date = bond_data['issue_date']
        d = maturity
        desc = [d]
        for _ in range(120):  # safety
            d = d - bond_data['payment_frequency']
            desc.append(d)
            if d <= until_date:
                break
        asc = sorted(set(desc))
        return asc[0]

    # розрахувати зміщену дату effectiveDate для вирівнювання 1-го купону
    start_date = coupon_dates_from_maturity(bond_data)    
    
    # Правильні конвенції для українського суверенного боргу
    # day_count = ql.ActualActual(ql.ActualActual.Bond) # day_count = ql.ActualActual(ql.ActualActual.ISMA) # day_count = ql.Actual364()
    day_count = ql.ActualActual(ql.ActualActual.ISMA)
    # day_count = ql.Actual365Fixed()
    calendar = ql.NullCalendar()
    business_convention = ql.Unadjusted

    # Точний розклад купонів
    schedule = ql.MakeSchedule(
        # effectiveDate=bond_data['issue_date'],
        effectiveDate=start_date,
        terminationDate=bond_data['maturity_date'],
        tenor=ql.Period(bond_data['payment_frequency'], ql.Days),
        calendar=calendar,
        convention=business_convention,
        terminalDateConvention=business_convention,
        rule=ql.DateGeneration.Backward, # Дуже важливо для ОВДП Мінфіну
        endOfMonth=False
    )

    # Ініціалізація грошових потоків та базового класу Bond # bond = ql.Bond(settlement_days, calendar, issue_date, fixed_rate_leg)
    fixed_rate_leg = ql.FixedRateLeg(
        schedule,
        day_count,
        [bond_data['face_value']],
        [bond_data['coupon_rate']]
    )

    settlement_days = 0
    bond = ql.FixedRateBond(
        settlement_days,
        bond_data['face_value'],
        schedule,
        [bond_data['coupon_rate']],
        day_count,
        business_convention
    )

    # Свторення явної кривої дисконтування та прив'язка bond до PricingEngine
    def get_dfs(calc_date, fixed_rate_leg, target_yield):
        dates = []
        dates.append(calc_date)
        discount_factors = []
        discount_factors.append(1.0)
        future = [d.date() for d in fixed_rate_leg if d.hasOccurred() == False]
        for d in future:
            days = d - calc_date
            t = max(0.0, days / 365.0)
            discount_factor = 1 / ((1.0 + target_yield) ** t)
            dates.append(d)
            discount_factors.append(discount_factor)
        return dates, discount_factors

    dates, discount_factors = get_dfs(calc_date, fixed_rate_leg, target_yield)
    discount_curve = ql.DiscountCurve(dates, discount_factors, day_count, calendar)
    term_structure_handle = ql.YieldTermStructureHandle(discount_curve)
    bond_engine = ql.DiscountingBondEngine(term_structure_handle)
    bond.setPricingEngine(bond_engine)

    # 6. Розрахунок та виведення результатів
    settlement_date = bond.settlementDate().ISO()
    
    
    dtm = bond_data['maturity_date'] - calc_date
    if dtm <= bond_data['payment_frequency']:
        compounding = ql.Simple # Важливо: використовуємо прості відсотки
        frequency = ql.NoFrequency # Частота виплат - один раз
        method = 'SIM'
    else:
        compounding = ql.Compounded # Важливо: використовуємо прості відсотки
        frequency = ql.Semiannual # Частота виплат - один раз
        method = 'YTM'

    dirty_price = round(bond.dirtyPrice() / 100 * bond_data['face_value'], 2)
    clean_price = round(bond.cleanPrice() / 100 * bond_data['face_value'], 2)
    dirty_price_simple = round(bond.dirtyPrice(target_yield, day_count, ql.Simple, ql.NoFrequency, calc_date) / 100 * bond_data['face_value'], 2)
    # dirty_price = round(bond.dirtyPrice(target_yield, day_count, compounding, frequency, calc_date) / 100 * bond_data['face_value'], 2)
    # clean_price = round(bond.cleanPrice(target_yield, day_count, compounding, frequency, calc_date) / 100 * bond_data['face_value'], 2)
    accrued_amount = round(bond.accruedAmount() / 100 * bond_data['face_value'], 2)

    result = {
        'target_yield': target_yield,
        'start_date':start_date.ISO(),
        'settlement_date': settlement_date,
        'method': method,
        'dirty_price': dirty_price,
        'dirty_price_simple': dirty_price_simple,
        'clean_price': clean_price,
        'accrued_amount': accrued_amount
    }

    # print(f"Дата розрахунку: {bond.settlementDate().ISO()}; Дата випуску: {bond_data['issue_date'].ISO()}; Теоретична дата початку: {bond_data['start_date'].ISO()}")
    # print(f"Дата розрахунку: {bond.settlementDate().ISO()}; isin: {bond_data['isin']}; Dirty: {bond.dirtyPrice() / 100 * bond_data['face_value']:.2f} UAH; НКД: {bond.accruedAmount() / 100 * bond_data['face_value']:.2f} | Clean: {bond.cleanPrice() / 100 * bond_data['face_value']:.2f}")
    if bond_data['isin'] == 'UA4000234413':
        print(result)
    
    # print()
    # print("Графік купонів:")
    # for cf in fixed_rate_leg:
    #     # if cf.hasOccurred() == False:
    #         coupon = ql.as_coupon(cf)
    #         if coupon:
    #             leg_item = ql.as_coupon(cf)
    #             start_date = leg_item.accrualStartDate()
    #             end_date = leg_item.accrualEndDate()
    #             days = end_date - start_date
    #             amount = leg_item.amount()
    #             print(f"{start_date.ISO()} | {end_date.ISO()} | {days} | {amount:.2f}")

    # print()
    # print("Дисконт-фактори:")
    # for node in discount_curve.nodes():
    #     print(f"{node[0].ISO()} | {node[1]}")
    
    return result
    # End function


In [ ]:
# deals
print("Обробку розпочато.")
dataframes = []
dataset = pd.read_excel(f"data/ua-bonds.xlsx", sheet_name="deals", engine='openpyxl')
for row in dataset.itertuples():
    bond_data = {
        'id': row.id,
        'deal_date': ql.Date(row.deal_date.day, row.deal_date.month, row.deal_date.year),
        'target_yield': round(float(row.ytm), 4) / 100, # round()
        'isin': row.ISIN,
        'face_value': 1000.0,
        'currency': row.Currency,
        'coupon_rate': float(row.Coupon_rate),
        'issue_date': ql.Date(row.Date_Issue.day, row.Date_Issue.month, row.Date_Issue.year),        
        'maturity_date': ql.Date(row.Date_maturity.day, row.Date_maturity.month, row.Date_maturity.year),
        'payment_frequency': int(row.pay_period),
        'coupon_per_year': 365 // int(row.pay_period),
        'bondtype': row.Type
    }
    if all([bond_data['maturity_date'] >= bond_data['deal_date']]):
        # print(f"   isin: {bond_data['isin']}")
        pfy_data = price_from_yield(bond_data['deal_date'], bond_data['target_yield'], bond_data)
        bond_data.update(pfy_data)
        bond_data['deal_date'] = bond_data['deal_date'].ISO()
        bond_data['issue_date'] = bond_data['issue_date'].ISO()
        bond_data['maturity_date'] = bond_data['maturity_date'].ISO()
        dataframes.append(bond_data)

result = pd.DataFrame(dataframes)
with pd.ExcelWriter(f"data/ua-bonds.xlsx", mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
    result.to_excel(writer, sheet_name="deals-pfy", index=False)
print("Обробку завершено.")

### Bond: DEV SIM Method

In [ ]:
import QuantLib as ql

# 1. Налаштування дат
today = ql.Date(8, ql.October, 2026)
ql.Settings.instance().evaluationDate = today

settlement_days = 0
settlement_date = today # Дата розрахунку (Т+0)
maturity_date = ql.Date(8, ql.October, 2027) # Погашення через 1 рік
day_counter = ql.Actual360()
face_amount = 100.0
coupon_rate = 0.05 # 5% річних
t = day_counter.yearFraction(settlement_date, maturity_date)
single_cashflow_amount = face_amount * (1.0 + coupon_rate * t) # Простий відсоток
cashflow = ql.SimpleCashFlow(single_cashflow_amount, maturity_date)
leg = ql.Leg([cashflow])
bond = ql.Bond(settlement_days, ql.NullCalendar(), settlement_date, leg)

market_yield = 0.04 # Очікувана доходність 4%
compounding = ql.Simple # Важливо: використовуємо прості відсотки
frequency = ql.Once # Частота виплат - один раз
dirty_price = bond.dirtyPrice(market_yield, day_counter, compounding, frequency, settlement_date)
print(f"Сума грошового потоку при погашенні: {single_cashflow_amount:.4f}")
print(f"Брутто-ціна (Dirty Price) облігації: {dirty_price:.4f}")

In [ ]:
import QuantLib as ql

# 1. Задаємо дати (наприклад, розрахунок від сьогодні до дати виплати через 6 місяців)
today = ql.Date(9, ql.October, 2026)
maturity_date = ql.Date(9, ql.April, 2027)

# 2. Визначаємо конвенцію підрахунку днів (наприклад, Actual/365 Fixed)
day_count = ql.Actual365Fixed()

# 3. Створюємо об'єкт процентної ставки:
# Параметри: ставка (5%), конвенція днів, тип відсотків (Simple), частота (зазвичай Annual)
rate_value = 0.05
interest_rate = ql.InterestRate(rate_value, day_count, ql.Simple, ql.Annual)

# 4. Розраховуємо дисконт-фактор (1 / (1 + r*T))
discount_factor = interest_rate.discountFactor(today, maturity_date)

# 5. Рахуємо ціну для майбутньої суми (наприклад, номінал 1000)
future_value = 1000.0
present_value = future_value * discount_factor

# Вивід результатів
year_fraction = day_count.yearFraction(today, maturity_date)
print(f"Частка року (T): {year_fraction:.4f}")
print(f"Дисконт-фактор: {discount_factor:.6f}")
print(f"Теперішня ціна (PV): {present_value:.2f}")
